# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [12]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [13]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [14]:
# TODO: Create a .env file with the following variables
# OPENAI_API_KEY="YOUR_KEY"
# CHROMA_OPENAI_API_KEY="YOUR_KEY"
# TAVILY_API_KEY="YOUR_KEY"
OPENAI_API_KEY="voc-27490172416886552967476a74cff6d447b5.06601520"
TAVILY_API_KEY="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"
OPENAI_BASE_URL="https://openai.vocareum.com/v1"


In [15]:
# TODO: Load environment variables
# load_dotenv()
from dotenv import load_dotenv
import os

load_dotenv("config.env")

if not os.getenv("OPENAI_API_KEY"):
 raise ValueError("OPENAI_API_KEY not found. Check your config.env file.")
assert os.getenv("TAVILY_API_KEY") is not None

OPENAI_BASE_URL = os.getenv(
    "OPENAI_BASE_URL",
    "https://openai.vocareum.com/v1"
)

### VectorDB Instance

In [19]:
# TODO: Instantiate your ChromaDB Client
# Choose any path you want
# chroma_client = chromadb.PersistentClient(path="chromadb")

client = chromadb.PersistentClient(path="./chromadb")


### Collection

In [24]:
# TODO: Pick one embedding function
# If picking something different than openai, 
# make sure you use the same when loading it
# embedding_fn = embedding_functions.OpenAIEmbeddingFunction()

embeddings_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("OPENAI_API_KEY"),
    api_base=os.getenv("OPENAI_BASE_URL")
   )

In [25]:
# TODO: Create a collection
# Choose any name you want
# collection = chroma_client.create_collection(
#    name="udaplay",
#    embedding_function=embedding_fn
#)

collection = client.get_or_create_collection(
    name="udaplay",
    embedding_function=embeddings_fn)

### Add documents

In [29]:
# Make sure you have a directory "project/starter/games"
data_dir = "games"

def build_metadata(game: dict) -> dict:
    return {
        "Name": str(game.get("Name", "")),
        "Platform": str(game.get("Platform", "")),
        "YearOfRelease": int(game["YearOfRelease"]) if game.get("YearOfRelease") else 0,
        "Genre": str(game.get("Genre", "")),
        "Publisher": str(game.get("Publisher", "")),
        "Description": str(game.get("Description", "")),
    }
if collection.count() == 0:
    ids, documents, metadatas = [], [], []

    for file_name in sorted(os.listdir(data_dir)):
        if not file_name.endswith(".json"):
            continue
        with open(os.path.join(data_dir, file_name), "r", encoding="utf-8") as f:
            game = json.load(f)

        content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"
        ids.append(os.path.splitext(file_name)[0])
        documents.append(content)
        metadatas.append(build_metadata(game))

    collection.add(ids=ids, documents=documents, metadatas=metadatas)
    print(f"Loaded {len(ids)} games into ChromaDB")  
 
else:
    print(f"Collection already populated with {collection.count()} games — skipping reload")  

Loaded 15 games into ChromaDB


In [30]:
# Demonstrate semantic search (required by the rubric)
def query_games(query: str, n_results: int = 3):
    results = collection.query(query_texts=[query], n_results=n_results)
    for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
        print(f"[{dist:.4f}] {meta['Name']} ({meta['Platform']}, {meta['YearOfRelease']}) — {meta['Publisher']}")
    return results

_ = query_games("open world game with a pirate theme")
_ = query_games("Nintendo platformer from the 90s")

[0.3778] Marvel's Spider-Man (PlayStation 4, 2018) — Sony Interactive Entertainment
[0.3850] Minecraft (Xbox One, 2014) — Mojang Studios
[0.4084] Grand Theft Auto: San Andreas (PlayStation 2, 2004) — Rockstar Games
[0.2613] Super Mario 64 (Nintendo 64, 1996) — Nintendo
[0.2697] Super Mario World (Super Nintendo Entertainment System (SNES), 1990) — Nintendo
[0.3637] Pokémon Gold and Silver (Game Boy Color, 1999) — Nintendo
